# Week 5: Your First Real ML Model — Linear Regression

## Student Practice Notebook

**Name:** Divya Saxena
**Roll Number:** AP24110011570
**Date:** 11-09-2026

## Learning Objectives

- Explain why preprocessing must happen *after* the train/test split, not before
- Fit a Simple Linear Regression model (one feature) with `sklearn`
- Fit a Multiple Linear Regression model (all features) with `sklearn`
- Compute and interpret R² and Adjusted R²
- Explain why Adjusted R² is a fairer comparison than R² when comparing models with different numbers of features

### Why this week matters
You now have clean, scaled, encoded data (Week 3) and know which features correlate with your target (Week 4). This week you finally build a model that predicts something. Everything after this — trees, SVM, neural nets — is the same core idea (fit → predict → evaluate) with a different equation underneath.

We use the **Cars24 used-car price dataset** throughout (`make`, `model`, `age`, `km_driven`, ... → target: `selling_price`).


## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
np.random.seed(1)


### Load the data


In [ ]:
!gdown 1bwRmKkPwmLKiqOgQ_LnKH0Vsc3mJKmVR

df = pd.read_csv('cars24-car-price-cleaned.csv')
df.head()


In [ ]:
df.shape


## Part 1 — Data Preparation (and a bug worth understanding)

### 1.1 A rushed way to preprocess — DO NOT do this, but let's see why

Here is how someone in a hurry might preprocess this data: encode the categorical columns, scale everything, **then** split into train/test.


In [ ]:
# --- THE WRONG WAY (run this so you can see the problem, then we fix it below) ---
from sklearn.preprocessing import MinMaxScaler

df_wrong = df.copy()
df_wrong['make'] = df_wrong.groupby('make')['selling_price'].transform('mean')
df_wrong['model'] = df_wrong.groupby('model')['selling_price'].transform('mean')

scaler_wrong = MinMaxScaler()
df_wrong_scaled = pd.DataFrame(scaler_wrong.fit_transform(df_wrong), columns=df_wrong.columns)
df_wrong_scaled.head()


### Predict, then think

**Before you read further: what do you think is wrong with the order of operations above?** Write your guess.

*Your answer:* Performing target encoding (calculating mean selling price by make and model) and min-max scaling on the entire dataset prior to splitting causes **data leakage**. Test set target prices and overall column min/max boundaries leak into training statistics. Consequently, the trained model gains unearned awareness of the test set, distorting evaluation metrics with artificially inflated accuracy.

---

**What's actually wrong:** both `groupby(...).transform('mean')` (target encoding) and `scaler.fit_transform(df)` were computed using the **entire dataset** — including rows that will later end up in the test set. That means:
- The mean price for each `make`/`model` "peeks" at test-set prices while encoding training rows.
- The min/max used for scaling includes test-set values.

This is called **data leakage**. Any model trained on this data will look better than it actually is, because it's indirectly seen the test set already. The fix is simple but easy to forget: **split first, then fit every transformer on the training data only, and apply (transform) it to the test data using those same training statistics.**


### 1.2 The correct order: split first, then fit on train only

In [ ]:
from sklearn.model_selection import train_test_split

# Split BEFORE any encoding or scaling touches the data
train_df, test_df = train_test_split(df, test_size=0.3, random_state=1)
train_df.shape, test_df.shape


### Student Practice — encode using train-only statistics

Using `train_df`, compute the mean `selling_price` per `make` and per `model` (same idea as the demo above, but only on `train_df`). Then **map** those same train-derived means onto both `train_df` and `test_df` — a test-set car's `make`/`model` gets the average price *learned from training data*, not from its own (unseen) price.

Hint: `train_df.groupby('make')['selling_price'].mean()` gives you a lookup table (a pandas Series indexed by `make`). Use `.map(...)` to apply that lookup to a column.


In [ ]:
# Encode 'make' and 'model' using train-only statistics
make_means = train_df.groupby('make')['selling_price'].mean()
train_df['make'] = train_df['make'].map(make_means)
test_df['make'] = test_df['make'].map(make_means)

model_means = train_df.groupby('model')['selling_price'].mean()
train_df['model'] = train_df['model'].map(model_means)
test_df['model'] = test_df['model'].map(model_means)

train_df.head()


### 1.3 Scaling — fit on train only, transform both

Same principle applies to scaling.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# fit_transform on TRAIN (learns min/max from train only)
train_scaled = pd.DataFrame(scaler.fit_transform(train_df), columns=train_df.columns, index=train_df.index)

# transform (NOT fit_transform) on TEST — reuses train's min/max
test_scaled = pd.DataFrame(scaler.transform(test_df), columns=test_df.columns, index=test_df.index)

train_scaled.head()


### Student Practice — build your final X/y splits

From `train_scaled` and `test_scaled`, create `X_train`, `y_train`, `X_test`, `y_test` (drop `selling_price` for X, keep it for y).


In [ ]:
# Create X and y splits for train and test sets
X_train = train_scaled.drop(columns=['selling_price'])
y_train = train_scaled['selling_price']

X_test = test_scaled.drop(columns=['selling_price'])
y_test = test_scaled['selling_price']

print('X_train shape:', X_train.shape, '| y_train shape:', y_train.shape)
print('X_test shape :', X_test.shape, ' | y_test shape :', y_test.shape)


## Part 2 — Simple Linear Regression

### 2.1 Demo: fit on a single feature (`km_driven`)


In [ ]:
from sklearn.linear_model import LinearRegression

X1_train = X_train[['km_driven']]
X1_test = X_test[['km_driven']]

uni_model = LinearRegression()
uni_model.fit(X1_train, y_train)

print('Coefficient (slope):', uni_model.coef_)
print('Intercept:', uni_model.intercept_)


In [ ]:
y_hat_uni = uni_model.predict(X1_test)

plt.figure(figsize=(6, 4))
plt.scatter(X1_test, y_test, label='actual', alpha=0.5)
plt.scatter(X1_test, y_hat_uni, color='orange', label='predicted', alpha=0.5)
plt.xlabel('km_driven (scaled)')
plt.ylabel('selling_price (scaled)')
plt.legend()
plt.show()


### Student Practice — try a different single feature

Pick a different single feature (e.g. `age` or `model`). Fit a `LinearRegression` on just that feature (train only), predict on the test set, and plot actual vs. predicted like the demo above.


In [ ]:
# Fit LinearRegression on a single feature 'age'
X_age_train = X_train[['age']]
X_age_test = X_test[['age']]

age_model = LinearRegression()
age_model.fit(X_age_train, y_train)

print('Coefficient (slope):', age_model.coef_[0])
print('Intercept:', age_model.intercept_)

y_hat_age = age_model.predict(X_age_test)

plt.figure(figsize=(6, 4))
plt.scatter(X_age_test, y_test, label='actual', alpha=0.5)
plt.scatter(X_age_test, y_hat_age, color='orange', label='predicted', alpha=0.5)
plt.xlabel('age (scaled)')
plt.ylabel('selling_price (scaled)')
plt.legend()
plt.show()


**Reflection:** Is the slope for your chosen feature positive or negative? Does that match your intuition (e.g., should an older car sell for more or less)?

*Your answer:* The slope for the `age` feature is negative. This matches intuition because as a vehicle gets older, its market value depreciates, resulting in a lower selling price.


## Part 3 — Multiple Linear Regression

### 3.1 Demo: fit using ALL features at once


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(X_train.columns, model.coef_):
    print(f'{feature:15s}: {coef:.4f}')
print('Intercept:', model.intercept_)


In [ ]:
y_hat = model.predict(X_test)

plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_hat, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='perfect prediction')
plt.xlabel('actual selling_price')
plt.ylabel('predicted selling_price')
plt.legend()
plt.show()


### Predict, then run

**Before running the cell below: do you expect the multiple-feature model's predictions to be closer to the diagonal line than the single-feature model's were? Why?**

*Your answer:* Yes, I expect the multiple-feature model's predictions to be much closer to the diagonal line. Selling price is influenced by multiple features simultaneously (age, distance driven, make, model, fuel type). A single feature captures only one component of variance, whereas using all relevant features provides the model with complete explanatory signal, reducing residual variance and resulting in a lower MSE.


In [ ]:
# Run this after writing your prediction above, then compare to what you expected
from sklearn.metrics import mean_squared_error

print('Single-feature test MSE:', mean_squared_error(y_test, y_hat_uni))
print('Multi-feature  test MSE:', mean_squared_error(y_test, y_hat))


## Part 4 — Evaluating Fit: R² and Adjusted R²

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

R² compares your model's error to a naive "always predict the average" baseline. 1.0 = perfect fit, 0 = no better than guessing the mean.

**Problem with plain R²:** it never decreases when you add more features — even useless, random ones. Adjusted R² penalizes features that don't earn their keep:

$$\bar{R}^2 = 1 - \frac{(1-R^2)(n-1)}{n-p-1}$$

where $n$ = number of samples, $p$ = number of features.


In [ ]:
def adj_r2_score(r_sq, n, p):
    return 1 - ((1 - r_sq) * (n - 1)) / (n - p - 1)

r2_uni = uni_model.score(X1_test, y_test)
r2_multi = model.score(X_test, y_test)

adj_r2_uni = adj_r2_score(r2_uni, n=X1_test.shape[0], p=1)
adj_r2_multi = adj_r2_score(r2_multi, n=X_test.shape[0], p=X_test.shape[1])

print(f'Single-feature  R2={r2_uni:.4f}   Adjusted R2={adj_r2_uni:.4f}')
print(f'Multi-feature   R2={r2_multi:.4f}   Adjusted R2={adj_r2_multi:.4f}')


### Student Practice — add a useless random feature

Add a column of pure random noise to `X_train`/`X_test` (e.g. `np.random.rand(len(X_train))`), refit `LinearRegression` on the expanded feature set, and compute both plain R² and Adjusted R² on the test set. Compare to the multi-feature model above.


In [ ]:
# Add a column of pure random noise to X_train and X_test
np.random.seed(42)
X_train_noise = X_train.copy()
X_test_noise = X_test.copy()

X_train_noise['random_noise'] = np.random.rand(len(X_train))
X_test_noise['random_noise'] = np.random.rand(len(X_test))

# Refit LinearRegression on expanded feature set
noise_model = LinearRegression()
noise_model.fit(X_train_noise, y_train)

# Compute plain R2 and Adjusted R2
r2_noise = noise_model.score(X_test_noise, y_test)
adj_r2_noise = adj_r2_score(r2_noise, n=X_test_noise.shape[0], p=X_train_noise.shape[1])

print(f'Multi-feature (original) R2={r2_multi:.4f}   Adjusted R2={adj_r2_multi:.4f}')
print(f'Multi-feature + Noise    R2={r2_noise:.4f}   Adjusted R2={adj_r2_noise:.4f}')


**Reflection:** Did plain R² go up, down, or stay about the same after adding the random feature? What about Adjusted R²? What does this tell you about trusting R² alone when comparing models with different numbers of features?

*Your answer:* Plain R² stayed about the same or slightly increased, as linear regression optimization can capture slight sample noise. In contrast, Adjusted R² decreased because of the degree-of-freedom penalty applied for adding extra features ($p$). This demonstrates that plain R² can be misleadingly optimistic when redundant or noisy features are introduced, whereas Adjusted R² properly accounts for feature count and model complexity.


## Summary

| What we did | Why it matters |
|---|---|
| Split before preprocessing | Prevents data leakage — your test score has to mean something |
| Simple Linear Regression | Baseline intuition: one feature, one line |
| Multiple Linear Regression | Uses all available signal |
| R² / Adjusted R² | R² alone can be gamed by adding useless features — Adjusted R² can't |

**Next week:** Polynomial Regression, and what happens when a model gets *too* flexible (overfitting) — plus how Ridge/Lasso/ElasticNet regularization fixes it.
